# Callbot Pipeline Test — ASR → LLM → TTS

Kết nối đầy đủ 3 service:
- **ASR**: Sherpa-ONNX Online Transducer (Vietnamese)
- **Brain (LLM + RAG)**: Qwen2.5-4B-Instruct qua vLLM + Qdrant
- **TTS**: VieNeu-TTS 0.3B

**Yêu cầu trên Drive** (`/content/drive/MyDrive/Callbot/`):
```
Callbot/
  asr_models/               ← 4 file model sherpa-onnx
    encoder-epoch-31-avg-11-chunk-16-left-128.fp16.onnx
    decoder-epoch-31-avg-11-chunk-16-left-128.fp16.onnx
    joiner-epoch-31-avg-11-chunk-16-left-128.fp16.onnx
    config.json
  qdrant_snapshot/           ← file .snapshot của Qdrant
    nutrition_articles-*.snapshot
```

> **Code repo** sẽ được `git clone` trực tiếp từ GitHub — không cần lưu trên Drive.

## 0. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Config — chỉnh ở đây

In [ ]:
import os

DRIVE = "/content/drive/MyDrive/Callbot"  # <-- đổi nếu khác

# ── Repo ─────────────────────────────────────────────────────────────
REPO_URL    = "https://github.com/enigmaticcat/legal-voice-callbot.git"
REPO_BRANCH = "main"                                                   # <-- đổi nếu cần

# ── Paths ────────────────────────────────────────────────────────────
ASR_MODEL_DIR = f"{DRIVE}/asr_models"          # 4 file .onnx + config.json
SNAPSHOT_PATH = next(
    (str(p) for p in __import__('pathlib').Path(f"{DRIVE}/qdrant_snapshot").glob("*.snapshot")),
    ""
)  # tự tìm file snapshot

# ── Models ───────────────────────────────────────────────────────────
VLLM_MODEL    = "Qwen/Qwen2.5-4B-Instruct"
EMBED_MODEL   = "intfloat/multilingual-e5-large-instruct"
COLLECTION    = "nutrition_articles"

# ── Ports ────────────────────────────────────────────────────────────
VLLM_PORT    = 8000   # vLLM OpenAI-compatible API
ASR_PORT     = 50051
BRAIN_PORT   = 50052
TTS_PORT     = 50053
GATEWAY_PORT = 8080   # Gateway (khác 8000 để tránh xung đột vLLM)

# ── vLLM ─────────────────────────────────────────────────────────────
GPU_MEM_UTIL  = "0.45"
MAX_MODEL_LEN = "8192"

# Kiểm tra snapshot
print(f"Repo URL    : {REPO_URL}")
print(f"Branch      : {REPO_BRANCH}")
print(f"ASR models  : {ASR_MODEL_DIR}")
print(f"Snapshot    : {SNAPSHOT_PATH or 'KHÔNG TÌM THẤY — kiểm tra lại Drive'}")
print(f"LLM model   : {VLLM_MODEL}")
print(f"Gateway port: {GATEWAY_PORT}")

## 2. Cài system dependencies

In [ ]:
# espeak-ng cho TTS phonemizer
!apt-get install -y -q espeak-ng libespeak-ng-dev

## 3. Cài Python dependencies

In [ ]:
# ASR
!pip install -q sherpa-onnx

# Brain
!pip install -q \
    fastapi uvicorn[standard] httpx openai \
    sentence-transformers qdrant-client \
    python-dotenv

# TTS
!pip install -q phonemizer soundfile librosa neucodec

# vLLM
!pip install -q vllm

# Gateway static files + tunnel
!pip install -q aiofiles pyngrok

# Notebook utils
!pip install -q ipython requests numpy

## 4. Clone repo về /content

In [ ]:
import os, subprocess

WORK_DIR = "/content/callbot"

if os.path.exists(WORK_DIR):
    subprocess.run(["rm", "-rf", WORK_DIR], check=True)

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, WORK_DIR],
    check=True,
)
print(f"Cloned → {WORK_DIR}")
!ls {WORK_DIR}

## 4.5. Build frontend

In [ ]:
import subprocess, sys

# Kiểm tra Node / npm
for cmd in [["node", "--version"], ["npm", "--version"]]:
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(f"{cmd[0]}: {r.stdout.strip() or 'NOT FOUND'}")

# Build frontend — không cần VITE_GATEWAY_URL vì URL được suy ra từ window.location
print("\nBuilding frontend (npm ci + npm run build)...")
subprocess.run(["npm", "ci", "--silent"], cwd=f"{WORK_DIR}/web", check=True)
subprocess.run(["npm", "run", "build"],   cwd=f"{WORK_DIR}/web", check=True)
print(f"Build done → {WORK_DIR}/web/dist")
!ls {WORK_DIR}/web/dist

## 5. Copy ASR model files

In [ ]:
import shutil, os

ASR_DATA_DIR = f"{WORK_DIR}/asr/data"
os.makedirs(ASR_DATA_DIR, exist_ok=True)

for fname in os.listdir(ASR_MODEL_DIR):
    src = os.path.join(ASR_MODEL_DIR, fname)
    dst = os.path.join(ASR_DATA_DIR, fname)
    shutil.copy2(src, dst)
    print(f"  Copied: {fname}")

print("\nASR models ready.")
!ls -lh {ASR_DATA_DIR}

## 6. Khởi động Qdrant + load snapshot

In [ ]:
import subprocess, time, requests, os

# Download Qdrant binary
!curl -sL https://github.com/qdrant/qdrant/releases/latest/download/qdrant-x86_64-unknown-linux-musl.tar.gz \
    -o /content/qdrant.tar.gz
!tar -xzf /content/qdrant.tar.gz -C /content/ qdrant
!chmod +x /content/qdrant

# Khởi động
qdrant_proc = subprocess.Popen(
    ["/content/qdrant"],
    stdout=open("/tmp/qdrant.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(5)

# Health check
r = requests.get("http://localhost:6333/healthz", timeout=5)
print(f"Qdrant: {r.status_code} (PID={qdrant_proc.pid})")

# Load snapshot
print(f"\nLoading snapshot: {SNAPSHOT_PATH}")
requests.delete("http://localhost:6333/collections/nutrition_articles")

with open(SNAPSHOT_PATH, "rb") as f:
    resp = requests.post(
        f"http://localhost:6333/collections/{COLLECTION}/snapshots/upload?priority=snapshot",
        files={"snapshot": f},
        timeout=600,
    )
assert resp.status_code == 200, f"Snapshot upload failed: {resp.text}"

# Chờ index
for _ in range(60):
    info = requests.get(f"http://localhost:6333/collections/{COLLECTION}").json()
    status = info["result"]["status"]
    count  = info["result"]["points_count"]
    if status == "green":
        print(f"Collection '{COLLECTION}': {count:,} vectors — OK")
        break
    time.sleep(3)

## 7. Khởi động vLLM — Qwen2.5-4B-Instruct

In [ ]:
import subprocess, sys, time, requests, os

os.system("pkill -f vllm.entrypoints 2>/dev/null")
time.sleep(2)

vllm_proc = subprocess.Popen(
    [
        sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--model",                  VLLM_MODEL,
        "--port",                   str(VLLM_PORT),
        "--max-model-len",          MAX_MODEL_LEN,
        "--gpu-memory-utilization", GPU_MEM_UTIL,
        "--trust-remote-code",
        # Qwen2.5 không có thinking mode — không cần enable_thinking flag
    ],
    stdout=open("/tmp/vllm.log", "w"),
    stderr=subprocess.STDOUT,
)

print(f"Chờ vLLM load {VLLM_MODEL} (~3-5 phút lần đầu)...")
for i in range(120):
    try:
        r = requests.get(f"http://localhost:{VLLM_PORT}/health", timeout=2)
        if r.status_code == 200:
            print(f"vLLM ready! ({i*5}s, PID={vllm_proc.pid})")
            break
    except:
        pass
    time.sleep(5)
    if i % 12 == 11:
        print(f"  {(i+1)*5}s — vẫn loading...")
else:
    print("TIMEOUT — xem log:")
    !tail -20 /tmp/vllm.log

## 8. Khởi động ASR server

In [ ]:
import subprocess, time, requests, os

asr_env = {
    **os.environ,
    "ASR_PORT":        str(ASR_PORT),
    "ASR_ENCODER_PATH": f"{WORK_DIR}/asr/data/encoder-epoch-31-avg-11-chunk-16-left-128.fp16.onnx",
    "ASR_DECODER_PATH": f"{WORK_DIR}/asr/data/decoder-epoch-31-avg-11-chunk-16-left-128.fp16.onnx",
    "ASR_JOINER_PATH":  f"{WORK_DIR}/asr/data/joiner-epoch-31-avg-11-chunk-16-left-128.fp16.onnx",
    "ASR_TOKENS_PATH":  f"{WORK_DIR}/asr/data/config.json",
}

asr_proc = subprocess.Popen(
    ["python", "server.py"],
    cwd=f"{WORK_DIR}/asr",
    env=asr_env,
    stdout=open("/tmp/asr.log", "w"),
    stderr=subprocess.STDOUT,
)

print("Chờ ASR server...")
for i in range(30):
    time.sleep(3)
    try:
        r = requests.get(f"http://localhost:{ASR_PORT}/health", timeout=2)
        if r.status_code == 200:
            print(f"ASR ready! ({i*3}s)")
            break
    except:
        pass
else:
    print("ASR timeout — xem log:")
    !tail -20 /tmp/asr.log

## 9. Khởi động TTS server

In [ ]:
import subprocess, time, requests, os

tts_env = {
    **os.environ,
    "TTS_PORT": str(TTS_PORT),
}

tts_proc = subprocess.Popen(
    ["python", "server.py"],
    cwd=f"{WORK_DIR}/tts",
    env=tts_env,
    stdout=open("/tmp/tts.log", "w"),
    stderr=subprocess.STDOUT,
)

# TTS load model lần đầu chậm (~2-3 phút)
print("Chờ TTS server load VieNeu model (~2-3 phút lần đầu)...")
for i in range(60):
    time.sleep(5)
    try:
        r = requests.get(f"http://localhost:{TTS_PORT}/health", timeout=2)
        if r.status_code == 200:
            print(f"TTS ready! ({i*5}s)")
            break
    except:
        pass
    if i % 6 == 5:
        print(f"  {(i+1)*5}s...")
else:
    print("TTS timeout — xem log:")
    !tail -30 /tmp/tts.log

## 10.5. Khởi động Gateway + mở ngrok

In [ ]:
import requests

services = [
    ("Qdrant",   "http://localhost:6333/healthz"),
    ("vLLM",     f"http://localhost:{VLLM_PORT}/health"),
    ("ASR",      f"http://localhost:{ASR_PORT}/health"),
    ("TTS",      f"http://localhost:{TTS_PORT}/health"),
    ("Brain",    f"http://localhost:{BRAIN_PORT}/health"),
    ("Gateway",  f"http://localhost:{GATEWAY_PORT}/health"),
]

print("=" * 40)
all_ok = True
for name, url in services:
    try:
        r = requests.get(url, timeout=3)
        status = "OK" if r.status_code == 200 else f"HTTP {r.status_code}"
    except Exception as e:
        status = f"FAIL ({e})"
        all_ok = False
    print(f"  {name:<10} {status}")
print("=" * 40)
if all_ok:
    print("Tất cả sẵn sàng!")
    try:
        print(f"  UI: {UI_URL}")
    except NameError:
        print("  (chạy cell 10.5 để lấy UI URL)")
else:
    print("Có service chưa ready — kiểm tra log")

## 10. Khởi động Brain server

In [ ]:
import subprocess, time, requests, os

brain_env = {
    **os.environ,
    # Dùng vLLM thay Gemini
    "LLM_BACKEND":  "openai",
    "LLM_BASE_URL": f"http://localhost:{VLLM_PORT}/v1",
    "LLM_MODEL":    VLLM_MODEL,
    "OPENAI_API_KEY": "EMPTY",
    # Qdrant local
    "QDRANT_URL":   "http://localhost:6333",
    "QDRANT_API_KEY": "",
    "QDRANT_COLLECTION": COLLECTION,
    # TTS
    "TTS_URL":      f"http://localhost:{TTS_PORT}",
    "BRAIN_PORT":   str(BRAIN_PORT),
}

brain_proc = subprocess.Popen(
    ["python", "-m", "brain.server"],
    cwd=WORK_DIR,
    env=brain_env,
    stdout=open("/tmp/brain.log", "w"),
    stderr=subprocess.STDOUT,
)

print("Chờ Brain server...")
for i in range(30):
    time.sleep(3)
    try:
        r = requests.get(f"http://localhost:{BRAIN_PORT}/health", timeout=2)
        if r.status_code == 200:
            print(f"Brain ready! ({i*3}s)")
            break
    except:
        pass
else:
    print("Brain timeout — xem log:")
    !tail -30 /tmp/brain.log

## 11. Health check tất cả services

In [ ]:
import requests

services = [
    ("Qdrant",  "http://localhost:6333/healthz"),
    ("vLLM",    f"http://localhost:{VLLM_PORT}/health"),
    ("ASR",     f"http://localhost:{ASR_PORT}/health"),
    ("TTS",     f"http://localhost:{TTS_PORT}/health"),
    ("Brain",   f"http://localhost:{BRAIN_PORT}/health"),
]

print("=" * 40)
all_ok = True
for name, url in services:
    try:
        r = requests.get(url, timeout=3)
        status = "OK" if r.status_code == 200 else f"HTTP {r.status_code}"
    except Exception as e:
        status = f"FAIL ({e})"
        all_ok = False
    print(f"  {name:<10} {status}")
print("=" * 40)
print("Tất cả sẵn sàng!" if all_ok else "Có service chưa ready — kiểm tra log")

## 12. Test 1: Text → Brain (LLM+RAG) → TTS → Audio

In [ ]:
import requests, time, io, wave
import numpy as np
from IPython.display import Audio, display

def query_pipeline(text: str) -> bytes:
    """Text → Brain/think/stream → collect text → TTS → PCM bytes"""
    print(f"Q: {text}")
    t0 = time.perf_counter()

    # ── Brain: stream LLM answer ──────────────────────────────────────
    resp = requests.post(
        f"http://localhost:{BRAIN_PORT}/think",
        json={"query": text, "session_id": "colab-test"},
        timeout=60,
    )
    answer = resp.json()["text"]
    brain_ms = (time.perf_counter() - t0) * 1000
    print(f"A: {answer[:200]}..." if len(answer) > 200 else f"A: {answer}")
    print(f"Brain: {brain_ms:.0f}ms")

    # ── TTS: text → PCM ───────────────────────────────────────────────
    t1 = time.perf_counter()
    tts_resp = requests.post(
        f"http://localhost:{TTS_PORT}/speak",
        json={"text": answer},
        timeout=120,
    )
    tts_ms = (time.perf_counter() - t1) * 1000
    total_ms = (time.perf_counter() - t0) * 1000
    print(f"TTS: {tts_ms:.0f}ms | Total: {total_ms:.0f}ms")

    return tts_resp.content  # WAV bytes


# ── Chạy test ─────────────────────────────────────────────────────────
question = "Tôi nên ăn gì để tăng cường hệ miễn dịch?"
wav_bytes = query_pipeline(question)

# Phát audio
display(Audio(wav_bytes, autoplay=True))

## 13. Test 2: Audio file → ASR → Brain → TTS → Audio

In [ ]:
import requests, time, wave, io
import numpy as np
from IPython.display import Audio, display

# ── Tạo file audio test (sine wave giả) hoặc dùng file thật từ Drive ──
# Để test với file thật, đặt đường dẫn vào TEST_AUDIO_PATH
TEST_AUDIO_PATH = None  # ví dụ: f"{DRIVE}/test_audio.wav"

def load_audio_as_pcm(path: str) -> bytes:
    """Load WAV file → PCM int16 bytes, resample sang 16kHz nếu cần"""
    import librosa, soundfile as sf
    audio, sr = librosa.load(path, sr=16000, mono=True)
    pcm = (audio * 32768).clip(-32768, 32767).astype(np.int16)
    return pcm.tobytes()

def full_pipeline(audio_pcm: bytes) -> bytes:
    """PCM audio → ASR → Brain → TTS → WAV bytes"""
    t0 = time.perf_counter()

    # ── ASR ───────────────────────────────────────────────────────────
    asr_resp = requests.post(
        f"http://localhost:{ASR_PORT}/transcribe",
        data=audio_pcm,
        headers={"Content-Type": "application/octet-stream"},
        timeout=30,
    )
    transcript = asr_resp.json()["text"]
    asr_ms = (time.perf_counter() - t0) * 1000
    print(f"ASR  ({asr_ms:.0f}ms): '{transcript}'")

    if not transcript.strip():
        print("ASR không nhận ra được âm thanh")
        return b""

    # ── Brain ─────────────────────────────────────────────────────────
    t1 = time.perf_counter()
    brain_resp = requests.post(
        f"http://localhost:{BRAIN_PORT}/think",
        json={"query": transcript, "session_id": "colab-full"},
        timeout=60,
    )
    answer = brain_resp.json()["text"]
    brain_ms = (time.perf_counter() - t1) * 1000
    print(f"Brain({brain_ms:.0f}ms): '{answer[:150]}...'" if len(answer) > 150 else f"Brain({brain_ms:.0f}ms): '{answer}'")

    # ── TTS ───────────────────────────────────────────────────────────
    t2 = time.perf_counter()
    tts_resp = requests.post(
        f"http://localhost:{TTS_PORT}/speak",
        json={"text": answer},
        timeout=120,
    )
    tts_ms  = (time.perf_counter() - t2) * 1000
    total_ms = (time.perf_counter() - t0) * 1000
    print(f"TTS  ({tts_ms:.0f}ms) | Total: {total_ms:.0f}ms")

    return tts_resp.content


# ── Chạy test ─────────────────────────────────────────────────────────
if TEST_AUDIO_PATH:
    pcm = load_audio_as_pcm(TEST_AUDIO_PATH)
    print(f"Loaded audio: {len(pcm):,} bytes ({len(pcm)/2/16000:.1f}s)\n")
else:
    # Không có file thật → dùng text pipeline thay thế
    print("TEST_AUDIO_PATH chưa set — dùng text pipeline thay thế\n")
    wav_bytes = query_pipeline("Vitamin D có trong thực phẩm nào?")
    display(Audio(wav_bytes, autoplay=False))

if TEST_AUDIO_PATH:
    wav = full_pipeline(pcm)
    if wav:
        display(Audio(wav, autoplay=True))

## 14. Test streaming: Brain audio pipeline (LLM → TTS realtime)

In [ ]:
import requests, time, numpy as np
from IPython.display import Audio, display

def query_audio_stream(text: str) -> bytes:
    """
    Dùng /pipeline/audio-stream endpoint:
    Brain stream LLM → chunker → TTS → PCM realtime
    """
    print(f"Q: {text}")
    t0 = time.perf_counter()
    pcm_chunks = []
    ttfc_ms = None

    with requests.post(
        f"http://localhost:{BRAIN_PORT}/pipeline/audio-stream",
        json={"query": text, "session_id": "colab-stream"},
        stream=True,
        timeout=120,
    ) as resp:
        for chunk in resp.iter_content(chunk_size=4800):
            if chunk:
                if ttfc_ms is None:
                    ttfc_ms = (time.perf_counter() - t0) * 1000
                    print(f"TTFC: {ttfc_ms:.0f}ms")
                # Bỏ qua JSON metadata cuối
                if chunk.strip().startswith(b"{"):
                    continue
                pcm_chunks.append(chunk)

    total_ms = (time.perf_counter() - t0) * 1000
    pcm = b"".join(pcm_chunks)
    duration_s = len(pcm) / 2 / 24000
    print(f"Total: {total_ms:.0f}ms | Audio: {duration_s:.1f}s")

    # Convert PCM int16 → numpy để phát
    audio = np.frombuffer(pcm, dtype=np.int16).astype(np.float32) / 32768.0
    return audio


audio_arr = query_audio_stream("Bổ sung canxi như thế nào cho bà bầu?")
display(Audio(audio_arr, rate=24000, autoplay=True))

## 15. Dừng tất cả services

In [ ]:
import os

for proc, name in [(vllm_proc, "vLLM"), (asr_proc, "ASR"),
                   (tts_proc, "TTS"), (brain_proc, "Brain"),
                   (qdrant_proc, "Qdrant")]:
    try:
        proc.terminate()
        proc.wait(timeout=5)
        print(f"  {name} stopped")
    except Exception as e:
        print(f"  {name} error: {e}")

os.system("pkill -f vllm.entrypoints 2>/dev/null")
print("Done.")